In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix

## 1. Introduction
Predict loan default (Loan_Status) and optimize decision threshold to minimize business cost.
False Positive cost = 1, False Negative cost = 5 (default is more costly).

In [ ]:
# Apni raw link yahan paste kar
url = "https://raw.githubusercontent.com/your-username/your-repo/main/train_u6lujuX_CVtuZ9i.csv"
df = pd.read_csv(url)
df.head()

In [ ]:
# Missing values
df['Gender'].fillna(df['Gender'].mode()[0], inplace=True)
df['Married'].fillna(df['Married'].mode()[0], inplace=True)
df['Dependents'].fillna(df['Dependents'].mode()[0], inplace=True)
df['Self_Employed'].fillna(df['Self_Employed'].mode()[0], inplace=True)
df['LoanAmount'].fillna(df['LoanAmount'].median(), inplace=True)
df['Loan_Amount_Term'].fillna(df['Loan_Amount_Term'].median(), inplace=True)
df['Credit_History'].fillna(df['Credit_History'].median(), inplace=True)

# Target encode
df['Loan_Status'] = df['Loan_Status'].map({'Y':1, 'N':0})

# Encode categorical
le = LabelEncoder()
df['Gender'] = le.fit_transform(df['Gender'])
df['Married'] = le.fit_transform(df['Married'])
df['Education'] = le.fit_transform(df['Education'])
df['Self_Employed'] = le.fit_transform(df['Self_Employed'])
df['Property_Area'] = le.fit_transform(df['Property_Area'])
df['Dependents'] = df['Dependents'].replace('3+', 3).astype(int)

# Features and target
X = df.drop(['Loan_ID', 'Loan_Status'], axis=1)
y = df['Loan_Status']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
y_prob = model.predict_proba(X_test)[:,1]

In [ ]:
thresholds = np.linspace(0, 1, 50)
fp_cost = 1
fn_cost = 5
best_threshold = 0.5
min_cost = float('inf')

for thresh in thresholds:
    y_pred = (y_prob >= thresh).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    total_cost = fp * fp_cost + fn * fn_cost
    if total_cost < min_cost:
        min_cost = total_cost
        best_threshold = thresh

print(f"Best threshold: {best_threshold:.2f}")
print(f"Minimum total business cost: {min_cost}")

In [ ]:
y_pred_opt = (y_prob >= best_threshold).astype(int)
cm = confusion_matrix(y_test, y_pred_opt)
print("Confusion Matrix with optimized threshold:\n", cm)
print("False Positives:", cm[0,1])
print("False Negatives:", cm[1,0])

## 2. Conclusion
By changing threshold from 0.5 to {best_threshold:.2f}, we reduced business cost. This helps bank avoid costly default mistakes.